Install and Load Required Packages

In [ ]:
# Run this in a Jupyter cell to install
!pip install polars
!pip install pandas numpy

import os
import requests
import pandas as pd
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from pathlib import Path


List of OMOP files

omop-condition-era-export.csv        
omop-drug-era-export.csv     
omop-observation-export.csv         
omop-procedure-occurrence-export.csv
omop-condition-occurrence-export.csv  
omop-drug-exposure.csv       
omop-observation-period-export.csv  
omop-visit-detail-export.csv
omop-death-export.csv                 
omop-measurement-export.csv  
omop-person-export.csv              
omop-visit-occurrence-export.csv

In [ ]:
# Check OMOP file
import pandas as pd

file_path = '/choruspilot/mgh/omop-measurement-export.csv'

try:
    # Read ONLY the first 1000 rows
    df_sample = pd.read_csv(file_path, nrows=1000)
    
    print("Successfully loaded sample!")
    print(df_sample.info())
    display(df_sample.head())
    print(df_sample[PROCEDURE_SOURCE_VALUE])

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Run SOFA and Sepsis-3 Final script

import warnings
warnings.filterwarnings('ignore') # Silences the dateutil formatting warning

import pandas as pd
import numpy as np

# FIX: Grouped imports cleanly at the top
from omop_utils import set_verbose
from omop_calc_sofa import compute_daily_sofa
from omop_calc_sepsis3 import compute_suspected_infection, evaluate_sepsis3

file_mapping = {
    'person': 'omop-person-export',
    'visit_occurrence': 'omop-visit-occurrence-export',
    'measurement': 'omop-measurement-export',
    'drug_exposure': 'omop-drug-exposure',
    'procedure_occurrence': 'omop-procedure-occurrence-export',
    'condition_occurrence': 'omop-condition-occurrence-export'
}

cdm = {}
print("Loading tables...")

for standard_name, file_name in file_mapping.items():
    file_path = f'/choruspilot/mgh/{file_name}.csv'
    
    # Load with nrows for testing. Remove this later for the full run!
    df = pd.read_csv(file_path, nrows=500000)
    
    # Force all column names to lowercase
    df.columns = df.columns.str.lower()

    # Parse as UTC, strip the timezone entirely, and FORCE nanosecond precision
    for col in df.columns:
        if 'date' in col or 'time' in col:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True).dt.tz_localize(None).astype('datetime64[ns]')

    cdm[standard_name] = df

# Handle the missing concept_ancestor safely
cdm['concept_ancestor'] = pd.DataFrame() 

print("Data loaded, database quirks patched, and dates matched! Computing SOFA scores...")

try:
    set_verbose(True)
     
    # FIX: Replaced ancestor_df with cdm['concept_ancestor']
    daily_sofa = compute_daily_sofa(cdm, cdm['concept_ancestor'])
    
    # FIX: Fill missing labs with 0 so the Total SOFA sums correctly
    comp_cols = ['resp_sofa', 'cardio_sofa', 'neuro_sofa', 'hepatic_sofa', 'renal_sofa', 'coag_sofa']
    daily_sofa[comp_cols] = daily_sofa[comp_cols].fillna(0)
    daily_sofa['total_sofa'] = daily_sofa[comp_cols].sum(axis=1)
    
    daily_sofa.to_csv('output_daily_sofa.csv', index=False)
    
    print("\nSUCCESS! Here are the first 5 rows of daily_sofa:")
    display(daily_sofa.head())
    print(f"Computed SOFA for {daily_sofa['person_id'].nunique()} patients")

    # ==========================================
    # SEPSIS-3 CALCULATION
    # ==========================================
    print("\nIdentifying suspected infections...")
    suspected_infections = compute_suspected_infection(cdm, cdm['concept_ancestor'])
    print(f"Found {len(suspected_infections)} suspected infection events.")
    
    if not suspected_infections.empty:
        print("\nEvaluating Sepsis-3 criteria...")
        # Pass in the daily_sofa table we just built
        sepsis3_results = evaluate_sepsis3(daily_sofa, suspected_infections, cdm, cdm['concept_ancestor'])
        
        if not sepsis3_results.empty:
            # Save the final output
            sepsis3_results.to_csv('output_sepsis3_cohort.csv', index=False)
            
            print("\n🎉 SUCCESS! Here is your final Sepsis-3 cohort:")
            display(sepsis3_results.head())
            
            sepsis_count = sepsis3_results['is_sepsis3'].sum()
            print(f"\nFinal Result: Identified {sepsis_count} patients meeting Sepsis-3 criteria!")
        else:
            print("\nNo patients met the full evaluation criteria for the acute window.")
    else:
        print("\nNo valid infection events found to evaluate.")
    
except Exception as e:
    print(f"\nAn error occurred: {e}")

In [ ]:
# Check Procedure Value

import pandas as pd

# 1. Grab the procedure table
proc_df = cdm['procedure_occurrence']

print("--- Top 20 Most Common Procedures ---")
print(proc_df['procedure_source_value'].value_counts().head(20))

print("\n--- Searching for 'Culture' in Procedures ---")
# 2. Search for any text containing 'culture', 'bld', or 'blood'
culture_mask = proc_df['procedure_source_value'].str.contains('culture|blood|bld', case=False, na=False)

if culture_mask.sum() > 0:
    print(proc_df[culture_mask]['procedure_source_value'].value_counts())
else:
    print("No procedures found containing the word 'culture' or 'blood'.")

In [ ]:
# The ID you want to investigate
target_id = 4046297  # (You can also try 4046279 if this comes up empty)
meas_df = cdm['measurement']

print(f"--- Investigating ID {target_id} in Measurement Table ---")

# 1. Check measurement_type_concept_id
if 'measurement_type_concept_id' in meas_df.columns:
    type_matches = meas_df[meas_df['measurement_type_concept_id'] == target_id]
    print(f"Found {len(type_matches)} records in 'measurement_type_concept_id'")
    if not type_matches.empty:
        display(type_matches[['person_id', 'measurement_datetime', 'measurement_type_concept_id', 'measurement_source_value']].head())

# 2. Check measurement_concept_id (often where the specific lab test is stored)
if 'measurement_concept_id' in meas_df.columns:
    concept_matches = meas_df[meas_df['measurement_concept_id'] == target_id]
    print(f"\nFound {len(concept_matches)} records in 'measurement_concept_id'")
    if not concept_matches.empty:
        display(concept_matches[['person_id', 'measurement_datetime', 'measurement_concept_id', 'measurement_source_value']].head())

print(meas_df['measurement_type_concept_id'].unique())